# MAIN NOTEBOOK

### RESNET-18 Base Model (unchanged)

In [1]:
# PyTorch core
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# torchvision (datasets, models, transforms)
import torchvision
import torchvision.transforms as transforms
from torchvision import datasets
from torchvision import models

# utilities
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

In [2]:
class CIFARResNet18(nn.Module):
    def __init__(self, num_classes=10):
        """
        ResNet-18 adapted for CIFAR-10

        notable changes from the default ImageNet version:
          - the first convolution is changed from 7x7 stride 2 to 3x3 stride 1
            because CIFAR-10 images are only 32x32
          - initial max-pooling layer is removed, since aggressive early
            downsampling is not appropriate for such small inputs
          - the final fully connected layer is changed to output 10 classes
        """
        super().__init__()

        # torchvision's ResNet-18 definition
        # weights=None means we are training from scratch
        self.model = models.resnet18(weights=None)

        # replace the first convolution layer for CIFAR-sized inputs
        self.model.conv1 = nn.Conv2d(
            in_channels=3,
            out_channels=64,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )

        # remove the initial max-pool layer
        self.model.maxpool = nn.Identity()

        # replace final classifier to align with CIFAR-10's 10 classes
        self.model.fc = nn.Linear(
            in_features=self.model.fc.in_features,
            out_features=num_classes
        )

    def forward(self, x):
        """
        Forward pass through the adapted ResNet-18.
        """
        return self.model(x)

In [ ]:
# check the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CIFARResNet18(num_classes=10).to(device)
print(model)

CIFARResNet18(
  (model): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): Identity()
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        